# Assignment 2: Pandas

Source: [NYC Motor Vehicle Collisions - Crashes](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95). The repository includes the first 6,000 records ordered by collision_id, retrieved on 4 October 2026, as an unmodified CSV snapshot. This is a subset of a real dataset, not invented data. Batch uniqueness has not been confirmed.

AI-assisted work; review the methods and follow course disclosure rules.

In [1]:
import importlib.util
required = {"pyarrow":"pyarrow", "tabulate":"tabulate", "matplotlib":"matplotlib", "imblearn":"imbalanced-learn"}
missing_packages = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    %pip install -q {" ".join(missing_packages)}

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter
from IPython.display import display
raw_path = Path("raw_crashes.csv")
if not raw_path.exists():
    from urllib.request import urlretrieve
    urlretrieve("https://data.cityofnewyork.us/resource/h9gi-nx95.csv?$limit=6000&$order=collision_id%20ASC", raw_path)

## 1. Load with explicit types and reduce memory

In [3]:
header = pd.read_csv(raw_path, nrows=0).columns
numeric_cols = [c for c in header if c.startswith("number_of_")] + ["latitude", "longitude"]
text_cols = [c for c in header if c not in numeric_cols and c != "crash_date"]
dtypes = {c: "float64" for c in numeric_cols}
dtypes.update({c: "string" for c in text_cols})
raw = pd.read_csv(raw_path, dtype=dtypes, parse_dates=["crash_date"])
assert raw.shape[0] >= 5000 and raw.shape[1] >= 8
assert raw[numeric_cols].isna().any().any()
df = raw.copy()
before_memory = df.memory_usage(deep=True).sum()
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], downcast="float")
for c in text_cols:
    if df[c].nunique(dropna=True) <= 100:
        df[c] = df[c].astype("category")
after_memory = df.memory_usage(deep=True).sum()
memory_saving = 100 * (before_memory - after_memory) / before_memory
print(f"Memory before: {before_memory:,} bytes; after: {after_memory:,} bytes")
print(f"Saving: {memory_saving:.2f}%")

Memory before: 7,129,785 bytes; after: 2,423,931 bytes
Saving: 66.00%


## 2. One profiling table

In [4]:
def mode_value(series):
    counts = series.value_counts(dropna=False)
    return counts.index[0] if len(counts) else None
def mode_frequency(series):
    counts = series.value_counts(dropna=False)
    return int(counts.iloc[0]) if len(counts) else 0
profile = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_count": df.isna().sum(), "missing_percent": df.isna().mean()*100, "unique_count": df.nunique(), "most_frequent_value": df.apply(mode_value), "frequency": df.apply(mode_frequency)})
display(profile)

,dtype,missing_count,missing_percent,unique_count,most_frequent_value,frequency
crash_date,datetime64[ns],0,0.000000,616,2012-07-06 00:00:00,28
crash_time,string,0,0.000000,990,14:00,83
borough,category,625,10.416667,1,MANHATTAN,5375
zip_code,category,625,10.416667,12,10013,2054
latitude,float32,625,10.416667,485,NaN,625
longitude,float32,625,10.416667,474,NaN,625
location,string,625,10.416667,501,<NA>,625
on_street_name,string,11,0.183333,167,CANAL STREET,631
off_street_name,string,11,0.183333,188,CANAL STREET,319
cross_street_name,category,5989,99.816667,11,NaN,5989


## 3. Duplicates

collision_id is the source record identifier. Keep the first occurrence of a repeated ID in this ordered snapshot; without an update timestamp, we cannot establish which conflicting record is newer.

In [5]:
exact_duplicates = int(df.duplicated().sum())
key_duplicates = int(df.duplicated(["collision_id"]).sum())
print("Exact duplicates:", exact_duplicates, "Duplicate IDs:", key_duplicates)
df = df.drop_duplicates("collision_id", keep="first").copy()

Exact duplicates: 0 Duplicate IDs: 0


## 4. Standardise vehicle labels

Normalise existing labels only. The before/after counts show how much cleaning this snapshot actually needs.

In [6]:
print("Before:")
display(df["vehicle_type_code1"].value_counts(dropna=False).head(20))
vehicle = df["vehicle_type_code1"].astype("string").str.strip().str.upper()
vehicle = vehicle.replace({"SPORT UTILITY / STATION WAGON":"SUV/STATION WAGON", "STATION WAGON/SPORT UTILITY VEHICLE":"SUV/STATION WAGON", "PASSENGER VEHICLE":"PASSENGER VEHICLE", "TAXI CAB":"TAXI", "PICK-UP TRUCK":"PICKUP TRUCK"})
df["vehicle_type_code1"] = vehicle.astype("category")
print("After:")
display(df["vehicle_type_code1"].value_counts(dropna=False).head(20))

Before:


vehicle_type_code1
PASSENGER VEHICLE                 2551
SPORT UTILITY / STATION WAGON     1147
TAXI                               648
VAN                                371
SMALL COM VEH(4 TIRES)             255
BUS                                217
OTHER                              190
LIVERY VEHICLE                     164
UNKNOWN                            147
LARGE COM VEH(6 OR MORE TIRES)     135
PICK-UP TRUCK                       96
MOTORCYCLE                          48
AMBULANCE                           13
FIRE TRUCK                           7
SCOOTER                              7
BICYCLE                              4
Name: count, dtype: int64

After:


vehicle_type_code1
PASSENGER VEHICLE                 2551
SUV/STATION WAGON                 1147
TAXI                               648
VAN                                371
SMALL COM VEH(4 TIRES)             255
BUS                                217
OTHER                              190
LIVERY VEHICLE                     164
UNKNOWN                            147
LARGE COM VEH(6 OR MORE TIRES)     135
PICKUP TRUCK                        96
MOTORCYCLE                          48
AMBULANCE                           13
FIRE TRUCK                           7
SCOOTER                              7
BICYCLE                              4
Name: count, dtype: int64

## 5. Extract a street number with a regular expression

This is the number embedded in the street name, not a building address. Named streets and missing strings legitimately have no match.

In [7]:
street_number = df["on_street_name"].astype("string").str.extract(r"(\d+)", expand=False)
df["street_number"] = pd.to_numeric(street_number, errors="coerce").astype("Float64")
print("Rows with no regex match:", int(street_number.isna().sum()))
display(df[["on_street_name", "street_number"]].head(10))

Rows with no regex match: 5875


,on_street_name,street_number
0,CANAL STREET,<NA>
1,BATTERY PLACE,<NA>
2,WATER STREET,<NA>
3,WEST STREET,<NA>
4,WEST STREET,<NA>
5,MAIDEN LANE,<NA>
6,WEST HOUSTON STREET,<NA>
7,VARICK STREET,<NA>
8,BROADWAY,<NA>
9,GREENE STREET,<NA>


## 6. Parse dates and derive calendar fields

In [8]:
date_strings = pd.read_csv(raw_path, usecols=["crash_date"], dtype="string")["crash_date"]
parsed = pd.to_datetime(date_strings, errors="coerce")
print("Nonmissing dates that failed parsing:", int((date_strings.notna() & parsed.isna()).sum()))
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")
df["year"] = df.crash_date.dt.year
df["month"] = df.crash_date.dt.month
df["day_of_week"] = df.crash_date.dt.dayofweek
df["quarter"] = df.crash_date.dt.quarter

Nonmissing dates that failed parsing: 0


## 7. Group summary with named aggregations

In [9]:
def injury_iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)
summary = df.groupby("borough", observed=True, dropna=False).agg(records=("collision_id", "count"), mean_injuries=("number_of_persons_injured", "mean"), max_injuries=("number_of_persons_injured", "max"), total_killed=("number_of_persons_killed", "sum"), injury_iqr=("number_of_persons_injured", injury_iqr))
display(summary)

,records,mean_injuries,max_injuries,total_killed,injury_iqr
borough,,,,,
MANHATTAN,5375,0.151814,6.0,3.0,0.0
<NA>,625,0.177600,5.0,1.0,0.0


## 8. Centre values within each group

transform returns a value for every original row and preserves its index.
A group summary has fewer rows and would require an extra join to align it.

In [10]:
df["injuries_minus_borough_mean"] = df.number_of_persons_injured - df.groupby("borough", observed=True, dropna=False).number_of_persons_injured.transform("mean")
display(df[["borough", "number_of_persons_injured", "injuries_minus_borough_mean"]].head())

,borough,number_of_persons_injured,injuries_minus_borough_mean
0,MANHATTAN,0.0,-0.151814
1,MANHATTAN,0.0,-0.151814
2,<NA>,0.0,-0.177600
3,MANHATTAN,0.0,-0.151814
4,MANHATTAN,0.0,-0.151814


## 9. Pivot, melt and reconstruct

The round trip preserves the aggregated pivot, including missing combinations. Aggregation itself cannot recover individual crash records.

In [11]:
pivot = df.pivot_table(index="borough", columns="vehicle_type_code1", values="number_of_persons_injured", aggfunc="mean", observed=True)
long = pivot.reset_index().melt(id_vars="borough", var_name="vehicle_type_code1", value_name="mean_injuries")
restored = long.pivot(index="borough", columns="vehicle_type_code1", values="mean_injuries").reindex(index=pivot.index, columns=pivot.columns)
assert np.allclose(pivot.to_numpy(), restored.to_numpy(), equal_nan=True)
print("Lossless pivot round trip:", True)
display(long.head())

Lossless pivot round trip: True


,borough,vehicle_type_code1,mean_injuries
0,MANHATTAN,AMBULANCE,0.000000
1,MANHATTAN,BICYCLE,0.666667
2,MANHATTAN,BUS,0.093264
3,MANHATTAN,FIRE TRUCK,0.000000
4,MANHATTAN,LARGE COM VEH(6 OR MORE TIRES),0.032000


## 10. Join behaviour and validation

To make the differences visible, omit two IDs from each side of a split. Inner keeps shared IDs; left keeps the left IDs; outer keeps the union. Duplicating a shared ID on both sides creates all pairings for that ID, which one_to_one validation rejects.

In [12]:
left = df[["collision_id", "crash_date", "borough"]].iloc[:-2].copy()
right = df[["collision_id", "number_of_persons_injured"]].iloc[2:].copy()
merge_counts = {}
for how in ["inner", "left", "outer"]:
    merged = left.merge(right, on="collision_id", how=how, validate="one_to_one")
    merge_counts[how] = len(merged)
print(merge_counts)
common = left.collision_id[left.collision_id.isin(right.collision_id)].iloc[0]
l2 = pd.concat([left, left[left.collision_id == common]], ignore_index=True)
r2 = pd.concat([right, right[right.collision_id == common]], ignore_index=True)
try:
    l2.merge(r2, on="collision_id", validate="one_to_one")
except pd.errors.MergeError as error:
    print("Expected validation failure:", error)
print("Rows when many_to_many is explicitly allowed:", len(l2.merge(r2, on="collision_id", validate="many_to_many")))

{'inner': 5996, 'left': 5998, 'outer': 6000}
Expected validation failure: Merge keys are not unique in either left or right dataset; not a one-to-one merge
Rows when many_to_many is explicitly allowed: 5999


## 11. Seven-record rolling average and rank

Seven rows means seven recorded crashes, not seven calendar days. collision_id breaks ties between crashes on the same date.

In [13]:
df = df.sort_values(["borough", "crash_date", "collision_id"], na_position="last").reset_index(drop=True)
group = df.groupby("borough", observed=True, dropna=False)
df["injuries_rolling7"] = group.number_of_persons_injured.transform(lambda x: x.rolling(7, min_periods=7).mean().rename(None))
df["injury_rank"] = group.number_of_persons_injured.rank(method="min", ascending=False)
display(df[["borough", "crash_date", "number_of_persons_injured", "injuries_rolling7", "injury_rank"]].head(10))

,borough,crash_date,number_of_persons_injured,injuries_rolling7,injury_rank
0,MANHATTAN,2012-07-01,0.0,NaN,683.0
1,MANHATTAN,2012-07-01,0.0,NaN,683.0
2,MANHATTAN,2012-07-01,0.0,NaN,683.0
3,MANHATTAN,2012-07-01,0.0,NaN,683.0
4,MANHATTAN,2012-07-01,0.0,NaN,683.0
5,MANHATTAN,2012-07-01,0.0,NaN,683.0
6,MANHATTAN,2012-07-01,0.0,0.000000,683.0
7,MANHATTAN,2012-07-01,0.0,0.000000,683.0
8,MANHATTAN,2012-07-01,1.0,0.142857,92.0
9,MANHATTAN,2012-07-01,4.0,0.714286,5.0


## 12. Three suspicious records

A suspicion is a reason to investigate, not proof that the record is false.

In [14]:
suspect_score = df[["latitude", "longitude", "borough"]].isna().sum(axis=1) + ((df.latitude == 0) | (df.longitude == 0)).astype(int)*3
suspicious = df.loc[suspect_score.nlargest(3).index]
display(suspicious[["collision_id", "crash_date", "borough", "latitude", "longitude"]])
for _, row in suspicious.iterrows():
    issues = []
    if pd.isna(row.latitude) or pd.isna(row.longitude): issues.append("coordinates are missing, so the location cannot be checked")
    if pd.isna(row.borough): issues.append("borough is missing")
    if row.latitude == 0 or row.longitude == 0: issues.append("zero coordinates are outside NYC")
    print(f"Collision {row.collision_id}: " + "; ".join(issues) + ".")

,collision_id,crash_date,borough,latitude,longitude
5375,24,2012-07-01,<NA>,NaN,NaN
5376,5333,2012-07-06,<NA>,NaN,NaN
5377,90,2012-07-07,<NA>,NaN,NaN


Collision 24: coordinates are missing, so the location cannot be checked; borough is missing.
Collision 5333: coordinates are missing, so the location cannot be checked; borough is missing.
Collision 90: coordinates are missing, so the location cannot be checked; borough is missing.


## 13. Export and compare files

The cleaned table retains unknown source values as missing and the expected first-six rolling gaps. No observations are invented to fill them. CSV does not preserve category/date types; Parquet does. These are one-run local load timings.

In [15]:
df.to_csv("cleaned_crashes.csv", index=False)
df.to_parquet("cleaned_crashes.parquet", index=False)
start = perf_counter()
read_csv = pd.read_csv("cleaned_crashes.csv")
csv_time = perf_counter() - start
start = perf_counter()
read_parquet = pd.read_parquet("cleaned_crashes.parquet")
parquet_time = perf_counter() - start
file_comparison = pd.DataFrame({"format":["CSV", "Parquet"], "bytes":[Path("cleaned_crashes.csv").stat().st_size, Path("cleaned_crashes.parquet").stat().st_size], "load_seconds":[csv_time, parquet_time]})
display(file_comparison)
assert read_csv.shape == read_parquet.shape == df.shape
source = "https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95"
readme = f"# Assignment 2: Pandas\n\nDataset: {source}\n\nUnmodified 6,000-row snapshot ordered by collision_id, retrieved 4 October 2026. Confirm that no classmate uses this source before submission.\n\nMemory before: {before_memory:,} bytes; after: {after_memory:,} bytes; saving: {memory_saving:.2f}%.\n\n" + file_comparison.to_markdown(index=False) + "\n\nRun the notebook in Colab with raw_crashes.csv uploaded. Dependencies: numpy, pandas, pyarrow, tabulate. Timings vary by runtime. AI-assisted work; reviewed code and executed outputs are included.\n"
Path("README.md").write_text(readme, encoding="utf-8")

,format,bytes,load_seconds
0,CSV,1728793,0.107305
1,Parquet,159892,0.205517


696